# 24 · Revision analyses on the saved checkpoints (inference only, no training)
**Runtime: T4 GPU** (about 2–3 h in total; every part is resumable). Read-only for all run folders and data. Writes only `results/p2_rev3/`.

**A. Code replacement with activation-independent donors (180 hybrid runs).** The bank holds the codes of all 240 unique
training images of the run's fold (no sampling with replacement). *const* = exact mean of these 240 codes (per cell for
patch-wise models). *swap*, draw r = 0..9: the donor of test case i is training image number
`default_rng(int(md5(f"{case_id}|{r}")[:8], 16)).integers(240)` of the fold's training ROI identifiers in sorted order, so the
donor depends only on the case identifier, the draw index and the fold, is the same for every model and seed, and does not
depend on activations or batch composition. Donor identities are stored.

**B. Finite shots with joint bitstrings (60 circuit runs).** For each code the circuit's 16-outcome computational-basis
distribution is computed exactly, N bitstrings are drawn from it (N = 100, 1,000, 10,000; three draws), and all four Z
expectations are computed from the same N bitstrings (N circuit preparations per code; 16N per patch-wise image).

**C. Prediction examples** (primary schedule, seed 0) chosen by a fixed rule (cell 6), saved as a figure.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, gc, hashlib
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
V2_DIR   = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
EV_CBIS  = f'{PROJECT}/results/p2_jit_eval/eval_per_image_all.csv'      # notebook 14 (exact, full precision)
EV_DMID  = f'{PROJECT}/results/p2_dmid_v2/eval'                         # notebook 20 (exact, full precision)
OUT      = f'{PROJECT}/results/p2_rev3'
SETS     = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val'}
for s in SETS:
    os.makedirs(f'{OUT}/replace/{s}', exist_ok=True); os.makedirs(f'{OUT}/shots_joint/{s}', exist_ok=True)
CODE_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{CODE_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    CODE_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (CODE_DIR, f'{PROJECT}/code'):
    if p not in sys.path: sys.path.insert(0, p)
import tensorflow as tf
import qseg.models, qseg2
from qseg.config import Config
cfg = Config(work_dir=OUT); B = cfg.batch_size
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 1 · Data (same arrays and order as notebooks 14, 20 and 23) + stored exact Dice for the checks
meta = pd.read_csv(f'{V2_DIR}/metadata.csv')
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XD = np.stack([_png(f'{V2_DIR}/images/{r}.png', False) for r in meta.roi_id]); YD = np.stack([_png(f'{V2_DIR}/masks/{r}.png', True) for r in meta.roi_id])
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
ref_c = pd.read_csv(EV_CBIS); ref_c = ref_c[ref_c.dataset == 'CBIS']
ref_d = pd.concat([pd.read_csv(f) for s in SETS for f in glob.glob(f'{EV_DMID}/{s}/*.csv')], ignore_index=True)
ref_d['dataset'] = 'DMID'
REF = pd.concat([ref_c, ref_d], ignore_index=True).set_index(['set', 'model', 'fold', 'seed', 'dataset', 'roi_id']).sort_index()
assert REF.index.is_unique, 'duplicate stored rows'
print('DMID', XD.shape, '| CBIS', XC.shape, '| stored rows', len(REF))
def dice_fast(Y, P):
    yt, yp = Y[..., 0] > 0.5, P[..., 0] >= cfg.threshold
    tp = (yt & yp).sum((1, 2)); s = yt.sum((1, 2)) + yp.sum((1, 2))
    return np.where(s == 0, 1.0, 2.0 * tp / np.maximum(s, 1))
def runs_of(root, models=None):
    out = []
    for d in sorted(glob.glob(f'{root}/*_fold*_seed*')):
        if not (os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')): continue
        s = json.load(open(f'{d}/summary.json'))
        if models is None or s['model'] in models: out.append((d, s))
    return out
HYB = ('g_q', 'g_cl', 'g_id', 'p_q', 'p_cl', 'p_id')

In [ ]:
# 2 · Replacement layer with explicit per-image codes + self-test
REPL_G = tf.Variable(tf.zeros((B, 4)), trainable=False); REPL_P = tf.Variable(tf.zeros((B, 4, 4, 4)), trainable=False)
USE = tf.Variable(0.0, trainable=False)
_sever_orig = qseg2.Sever.call
def _sever_call(self, x):
    y = _sever_orig(self, x)
    r = REPL_G if len(x.shape) == 2 else REPL_P
    return tf.where(USE > 0.5, tf.cast(r[:tf.shape(x)[0]], y.dtype), y)
qseg2.Sever.call = _sever_call
def load(d, s):
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(s['model'], (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{d}/best.weights.h5')
    try: m.jit_compile = False
    except Exception: pass
    return m
def forward(m):
    f = tf.function(lambda x: m(x, training=False), reduce_retracing=True)
    def run(X, repl=None):
        out = []
        for i in range(0, len(X), B):
            xb = X[i:i + B]
            if repl is None: USE.assign(0.0)
            else:
                r = np.zeros((B,) + repl.shape[1:], np.float32); r[:len(xb)] = repl[i:i + B]
                (REPL_G if repl.ndim == 2 else REPL_P).assign(r); USE.assign(1.0)
            out.append(f(tf.constant(xb)).numpy())
        USE.assign(0.0); return np.concatenate(out)
    return run
def codes_of(m, X):
    cm = tf.keras.Model(m.input, m.get_layer('sever').input)
    return np.concatenate([cm(tf.constant(X[i:i + B]), training=False).numpy() for i in range(0, len(X), B)]).astype(np.float32)
def donor(case_id, r, n):
    return int(np.random.default_rng(int(hashlib.md5(f'{case_id}|{r}'.encode()).hexdigest()[:8], 16)).integers(n))
# self-test: (1) normal forward reproduces the stored Dice, (2) replacing every code by the image's own code changes nothing
d0, s0 = [x for x in runs_of(SETS['train'], ('p_q',))][0]; m = load(d0, s0); run = forward(m)
_, _, te = qseg2.fold_split(DFC, int(s0['fold']))
P0 = run(XC[te]); own = codes_of(m, XC[te]); P1 = run(XC[te], own)
st = REF.loc[('train', s0['model'], int(s0['fold']), int(s0['seed']), 'CBIS')].dice.reindex(DFC.roi_id.values[te]).values
print(os.path.basename(d0), '| normal vs stored Dice, max |diff|:', float(np.max(np.abs(dice_fast(YC[te], P0) - st))),
      '| own-code replacement vs normal, max |dprob|:', float(np.abs(P1 - P0).max()))
assert float(np.abs(P1 - P0).max()) < 1e-4
del m; gc.collect()

In [ ]:
# 3 · A: code replacement, 240-image bank, activation-independent donors (180 hybrid runs; resumable)
K = 10
def eval_replace(set_name, d, s):
    dst = f'{OUT}/replace/{set_name}/{os.path.basename(d)}.csv'
    if os.path.exists(dst): return None
    m = load(d, s); run = forward(m); fold = int(s['fold'])
    tr, _, te = qseg2.fold_split(DFC, fold)
    order = np.argsort(DFC.roi_id.values[tr]); tr = tr[order]; tr_ids = DFC.roi_id.values[tr]   # training ROIs, sorted by identifier
    bank = codes_of(m, XC[tr]); const = bank.mean(0, keepdims=True)
    parts = []
    for ds, X, Y, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, meta.roi_id.values)):
        P0 = run(X); b0 = P0[..., 0] >= cfg.threshold
        t = pd.DataFrame({'dataset': ds, 'roi_id': ids, 'dice': dice_fast(Y, P0)})
        Pc = run(X, np.repeat(const, len(X), 0))
        t['dice_const'] = dice_fast(Y, Pc); t['flip_const'] = ((Pc[..., 0] >= cfg.threshold) != b0).mean((1, 2)); t['absdp_const'] = np.abs(Pc - P0)[..., 0].mean((1, 2))
        fl, ab = [], []
        for r in range(K):
            idx = np.array([donor(i, r, len(tr_ids)) for i in ids])
            Ps = run(X, bank[idx])
            t[f'dice_swap{r}'] = dice_fast(Y, Ps); t[f'donor{r}'] = tr_ids[idx]
            fl.append(((Ps[..., 0] >= cfg.threshold) != b0).mean((1, 2))); ab.append(np.abs(Ps - P0)[..., 0].mean((1, 2)))
        t['flip_swap'] = np.mean(fl, 0); t['absdp_swap'] = np.mean(ab, 0)
        parts.append(t)
    t = pd.concat(parts, ignore_index=True)
    t['model'], t['fold'], t['seed'], t['set'] = s['model'], fold, int(s['seed']), set_name
    t['bank_unique'] = len(np.unique(tr_ids))
    t.to_csv(dst, index=False, float_format='%.17g'); del m; gc.collect(); return t
n_new = 0
for set_name, root in SETS.items():
    rs = runs_of(root, HYB); print(set_name, len(rs), 'hybrid runs')
    for d, s in rs:
        t0 = time.time(); t = eval_replace(set_name, d, s)
        if t is not None:
            n_new += 1; g = t.groupby('dataset')
            print(f"{set_name:5s} {os.path.basename(d):22s} " + ' | '.join(
                f"{k} {v.dice.mean():.4f} const-drop {(v.dice - v.dice_const).mean():.4f} swap-drop {(v.dice - v[[f'dice_swap{r}' for r in range(K)]].mean(1)).mean():.4f}"
                for k, v in g) + f" ({time.time() - t0:.0f} s)", flush=True)
print('evaluated', n_new, 'new runs')

In [ ]:
# 4 · B: finite shots with joint bitstrings (60 circuit runs; resumable)
from qseg.quantum_sim import _ring_perms
def circuit_probs(x, w, n=4):
    # identical gate sequence to qseg.quantum_sim.circuit_expvals, returning the 2^n outcome probabilities (wire 0 = most significant bit)
    L = '_acdefghi'[1:n + 1]; theta = x * (np.pi / 2.0)
    qs = [tf.stack([tf.cos(theta[:, i] / 2), tf.sin(theta[:, i] / 2)], axis=1) for i in range(n)]
    psi = tf.cast(tf.einsum(','.join('b' + c for c in L) + '->b' + L, *qs), tf.complex64)
    perms = _ring_perms(n)
    for l in range(w.shape[0]):
        for i in range(n):
            phi, th, om = w[l, i, 0], w[l, i, 1], w[l, i, 2]
            c, s_ = tf.cos(th / 2), tf.sin(th / 2); a, b = (phi + om) / 2, (phi - om) / 2
            u = tf.stack([tf.stack([tf.complex(tf.cos(a) * c, -tf.sin(a) * c), tf.complex(-tf.cos(b) * s_, -tf.sin(b) * s_)]),
                          tf.stack([tf.complex(tf.cos(b) * s_, -tf.sin(b) * s_), tf.complex(tf.cos(a) * c, tf.sin(a) * c)])])
            psi = tf.reshape(tf.einsum('z' + L[i] + ',b' + L + '->b' + L[:i] + 'z' + L[i + 1:], u, psi), [-1] + [2] * n)
        flat = tf.reshape(psi, [-1, 2 ** n])
        for p in perms: flat = tf.gather(flat, p, axis=1)
        psi = tf.reshape(flat, [-1] + [2] * n)
    return tf.reshape(tf.math.real(psi * tf.math.conj(psi)), [-1, 2 ** n])
ZSIGN = tf.constant([[1.0 - 2.0 * ((k >> (3 - i)) & 1) for i in range(4)] for k in range(16)], tf.float32)   # (16, 4)
SHOTS = tf.Variable(0, dtype=tf.int32, trainable=False); DRAW = tf.Variable(0, dtype=tf.int64, trainable=False)
_exact_call = qseg.models.TFCircuitLayer.call
def _joint_call(self, inputs):
    e = _exact_call(self, inputs)
    def sample():
        x = tf.reshape(tf.cast(inputs, tf.float32), [-1, 4])
        p = circuit_probs(x, tf.cast(self.qweights, tf.float32))
        s2 = tf.math.floormod(tf.cast(tf.reduce_sum(tf.abs(inputs)) * 1000.0, tf.int64), 1_000_003)
        k = tf.random.stateless_categorical(tf.math.log(tf.maximum(p, 1e-30)), SHOTS, seed=tf.stack([DRAW, s2]))   # (rows, N) outcomes
        counts = tf.reduce_sum(tf.one_hot(k, 16, dtype=tf.float32), axis=1)                                      # (rows, 16)
        est = tf.matmul(counts, ZSIGN) / tf.cast(SHOTS, tf.float32)
        return tf.reshape(est, tf.shape(e))
    return tf.cond(SHOTS > 0, sample, lambda: e)
qseg.models.TFCircuitLayer.call = _joint_call
assert qseg2.PatchCircuitLayer.__mro__[1] is qseg.models.TFCircuitLayer
LEVELS, ND = (100, 1000, 10000), 3
def set_shots(n, r=0): SHOTS.assign(int(n)); DRAW.assign(int(5000 + 10 * np.log10(max(n, 1)) + r) if n > 0 else 0)
# self-test: exact expectations from the probabilities equal the simulator; shot estimates lie on the 2k/N - 1 grid; joint sampling
d0, s0 = runs_of(SETS['train'], ('p_q',))[0]; m = load(d0, s0); lay = m.get_layer('quantum_circuit')
xin = tf.random.uniform((64, 4), -1, 1, seed=1)
e_sim = _exact_call(lay, xin).numpy(); e_prob = (circuit_probs(xin, tf.cast(lay.qweights, tf.float32)) @ ZSIGN).numpy()
print('max |simulator - probabilities|:', float(np.abs(e_sim - e_prob).max())); assert float(np.abs(e_sim - e_prob).max()) < 1e-5
for n in LEVELS:
    set_shots(n); est = lay(xin).numpy(); set_shots(0)
    k = (est + 1) * n / 2; print(f'N={n:5d}: on grid {np.allclose(k, np.round(k), atol=1e-2)} | mean |shot - exact| {np.abs(est - e_sim).mean():.4f}')
del m; gc.collect()
def eval_joint(set_name, d, s):
    dst = f'{OUT}/shots_joint/{set_name}/{os.path.basename(d)}.csv'
    if os.path.exists(dst): return None
    m = load(d, s); _, _, te = qseg2.fold_split(DFC, int(s['fold'])); parts = []
    for ds, X, Y, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, meta.roi_id.values)):
        set_shots(0); P0 = m.predict(X, batch_size=B, verbose=0); b0 = P0[..., 0] >= cfg.threshold
        t = pd.DataFrame({'dataset': ds, 'roi_id': ids, 'dice_exact': dice_fast(Y, P0)})
        for n in LEVELS:
            for r in range(ND):
                set_shots(n, r); Pn = m.predict(X, batch_size=B, verbose=0)
                t[f'dice_s{n}_d{r}'] = dice_fast(Y, Pn); t[f'flip_s{n}_d{r}'] = ((Pn[..., 0] >= cfg.threshold) != b0).mean((1, 2))
        set_shots(0); parts.append(t)
    t = pd.concat(parts, ignore_index=True)
    t['model'], t['fold'], t['seed'], t['set'] = s['model'], int(s['fold']), int(s['seed']), set_name
    t.to_csv(dst, index=False, float_format='%.17g'); del m; gc.collect(); return t
n_new = 0
for set_name, root in SETS.items():
    rs = runs_of(root, ('g_q', 'p_q')); print(set_name, len(rs), 'circuit runs')
    for d, s in rs:
        t0 = time.time(); t = eval_joint(set_name, d, s)
        if t is not None:
            n_new += 1; g = t.groupby('dataset')
            print(f"{set_name:5s} {os.path.basename(d):20s} " + ' | '.join(f"{k} exact {v.dice_exact.mean():.4f} s100 {v.dice_s100_d0.mean():.4f} s1000 {v.dice_s1000_d0.mean():.4f}" for k, v in g) + f" ({time.time() - t0:.0f} s)", flush=True)
print('evaluated', n_new, 'new runs')
qseg.models.TFCircuitLayer.call = _exact_call; set_shots(0)

In [ ]:
# 5 · C: prediction examples (primary schedule, seed 0). Selection rule, fixed before drawing:
#   CBIS-DDSM: each crop scored by the model of its own test fold; DMID: the fold-0 models.
#   m = mean Dice of the five shown models (g_q, g_cl, p_q, p_cl, unet) per crop.
#   rows: CBIS 50th and 10th percentile of m; DMID 50th and 10th percentile of m; DMID crop with the largest g_q - p_q Dice difference.
#   (nearest crop to each percentile; ties broken by identifier)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt, cv2
SHOW = ['g_q', 'g_cl', 'p_q', 'p_cl', 'unet']
rc = REF.reset_index(); rc = rc[(rc.set == 'train') & (rc.seed == 0) & rc.model.isin(SHOW)]
wc = rc[rc.dataset == 'CBIS'].pivot_table(index='roi_id', columns='model', values='dice')
wd = rc[(rc.dataset == 'DMID') & (rc.fold == 0)].pivot_table(index='roi_id', columns='model', values='dice')
def at_pct(w, q):
    mm = w[SHOW].mean(1).sort_index(); v = np.percentile(mm.values, q); return (mm - v).abs().sort_values(kind='stable').index[0]
rows = [('CBIS', at_pct(wc, 50), 'CBIS-DDSM, median case'), ('CBIS', at_pct(wc, 10), 'CBIS-DDSM, 10th percentile'),
        ('DMID', at_pct(wd, 50), 'DMID, median case'), ('DMID', at_pct(wd, 10), 'DMID, 10th percentile'),
        ('DMID', (wd.g_q - wd.p_q).sort_index().sort_values(ascending=False, kind='stable').index[0], 'DMID, largest G - P difference (circuit)')]
fold_of = dict(zip(DFC.roi_id, DFC.fold)); dmid_ix = {r: i for i, r in enumerate(meta.roi_id)}; cbis_ix = {r: i for i, r in enumerate(DFC.roi_id)}
need = {}
for ds, rid, _ in rows: need.setdefault(fold_of[rid] if ds == 'CBIS' else 0, []).append((ds, rid))
PRED = {}; RUNS0 = {(s['model'], int(s['fold'])): (d, s) for d, s in runs_of(SETS['train'], SHOW) if int(s['seed']) == 0}
for fold, items in need.items():
    for mdl in SHOW:
        dd, s = RUNS0[(mdl, fold)]; m = load(dd, s)
        X = np.stack([XC[cbis_ix[r]] if ds == 'CBIS' else XD[dmid_ix[r]] for ds, r in items])
        P = m.predict(X, batch_size=B, verbose=0)
        for (ds, r), p in zip(items, P): PRED[(r, mdl)] = p[..., 0]
        del m; gc.collect()
def outline(ax, img, ref, pred=None):
    rgb = np.repeat((img[..., None] * 255).astype(np.uint8), 3, -1).copy()
    cs, _ = cv2.findContours(ref.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(rgb, cs, -1, (255, 40, 40), 2)
    if pred is not None:
        cs, _ = cv2.findContours(pred.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(rgb, cs, -1, (255, 230, 0), 2)
    ax.imshow(rgb); ax.set_xticks([]); ax.set_yticks([])
fig, axs = plt.subplots(len(rows), 1 + len(SHOW), figsize=(2.1 * (1 + len(SHOW)), 2.2 * len(rows)))
cases = []
for i, (ds, rid, lab) in enumerate(rows):
    img = (XC[cbis_ix[rid]] if ds == 'CBIS' else XD[dmid_ix[rid]])[..., 0]; ref = (YC[cbis_ix[rid]] if ds == 'CBIS' else YD[dmid_ix[rid]])[..., 0] > 0.5
    outline(axs[i, 0], img, ref); axs[i, 0].set_ylabel(lab, fontsize=7)
    if i == 0: axs[i, 0].set_title('Crop and reference', fontsize=8)
    for j, mdl in enumerate(SHOW):
        pb = PRED[(rid, mdl)] >= cfg.threshold; dsc = float(dice_fast(ref[None, ..., None].astype(np.float32), PRED[(rid, mdl)][None, ..., None]))
        outline(axs[i, j + 1], img, ref, pb)
        axs[i, j + 1].set_title((f'{mdl}\n' if i == 0 else '') + (f'Dice {dsc:.2f}' + (' (empty)' if pb.sum() == 0 else '')), fontsize=7)
        cases.append(dict(row=i, dataset=ds, roi_id=rid, label=lab, model=mdl, fold=fold_of[rid] if ds == 'CBIS' else 0, dice=dsc, empty=bool(pb.sum() == 0)))
plt.tight_layout(); fig.savefig(f'{OUT}/fig_examples.png', dpi=300); fig.savefig(f'{OUT}/fig_examples.pdf'); plt.close(fig)
pd.DataFrame(cases).to_csv(f'{OUT}/fig_examples_cases.csv', index=False)
print(pd.DataFrame(cases).pivot_table(index=['row', 'roi_id'], columns='model', values='dice').round(3).to_string())

In [ ]:
# 6 · Combine + checks + summary
ra = pd.concat([pd.read_csv(f) for s in SETS for f in sorted(glob.glob(f'{OUT}/replace/{s}/*.csv'))], ignore_index=True)
ra.to_csv(f'{OUT}/replacement_v2_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
k = ['set', 'model', 'fold', 'seed', 'dataset', 'roi_id']
chk = ra.merge(REF.reset_index()[k + ['dice']].rename(columns={'dice': 'dice_stored'}), on=k, how='left')
print('A runs:', ra.groupby(['set', 'model', 'fold', 'seed']).ngroups, '| rows:', len(ra), '| unmatched:', int(chk.dice_stored.isna().sum()),
      '| max |normal - stored|:', float(np.nanmax(np.abs(chk.dice - chk.dice_stored))), '| bank sizes:', sorted(ra.bank_unique.unique()))
ra['drop_const'] = ra.dice - ra.dice_const; ra['drop_swap'] = ra.dice - ra[[f'dice_swap{r}' for r in range(K)]].mean(1)
print(ra.groupby(['set', 'dataset', 'model'])[['dice', 'drop_const', 'drop_swap']].mean().round(4).to_string())
sj = pd.concat([pd.read_csv(f) for s in SETS for f in sorted(glob.glob(f'{OUT}/shots_joint/{s}/*.csv'))], ignore_index=True)
sj.to_csv(f'{OUT}/shots_joint_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
for n in LEVELS: sj[f'delta_s{n}'] = sj[[f'dice_s{n}_d{r}' for r in range(ND)]].mean(1) - sj.dice_exact
print('B runs:', sj.groupby(['set', 'model', 'fold', 'seed']).ngroups, '| rows:', len(sj))
print(sj.groupby(['set', 'dataset', 'model'])[['dice_exact'] + [f'delta_s{n}' for n in LEVELS]].mean().round(4).to_string())
print('files:', sorted(f for f in os.listdir(OUT) if '.' in f))
print('DONE')

In [ ]:
# 7 · Auto-shutdown: save Drive, then release the GPU
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()